<h1>Эксперименты на одном Catboost</h1>
Тут можно поделать разное с фичами, чтобы потом раскатить на ансамбль

In [144]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='darkgrid')

In [295]:
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from catboost import CatBoostClassifier
import optuna

In [327]:
X_train = pd.read_csv('../data/train_X.csv').fillna(0) 
test_cv = pd.read_csv('../data/test1_X.csv').fillna(0)
Y_train = pd.read_csv('../data/train_y.csv')['default6']

In [328]:
X_train.columns[X_train.columns.str.startswith('agg_spark_extended')]

Index(['agg_spark_extended_report__g_contractor__EstimatedClaimsSum__last__ALL_TIME',
       'agg_spark_extended_report__g_contractor__EstimatedLiabilitiesSum__last__ALL_TIME',
       'agg_spark_extended_report__g_contractor__EstimatedNetLiabilitiesSum__last__ALL_TIME',
       'agg_spark_extended_report__g_contractor__PledgeeActiveCount__last__ALL_TIME',
       'agg_spark_extended_report__g_contractor__PledgeeCeasedCount__last__ALL_TIME',
       'agg_spark_extended_report__g_contractor__PledgerActiveCount__last__ALL_TIME',
       'agg_spark_extended_report__g_contractor__PledgerCeasedCount__last__ALL_TIME',
       'agg_spark_extended_report__g_contractor__CompanySizeRevenue__last__ALL_TIME',
       'agg_spark_extended_report__g_contractor__CreditLimitSum__last__ALL_TIME'],
      dtype='object')

In [329]:
groups = X_train['contract_id']
X_train = X_train.drop(['report_date', 'contract_date', 'contract_id', 'specialization_id', 'project_id', 'contractor_id', 'building_id'], axis=1) #[['contract_init_sum', 'specialization_id', 'project_id', 'contractor_id']]
test_cv = test_cv.drop(['report_date', 'contract_date', 'contract_id'], axis=1)

In [ ]:
X_train['cluster'] = KMeans(n_clusters=4).fit_predict(X_train['contract_current_sum'].values.reshape(-1,1))

In [330]:
X_train.head()

,contract_init_sum,contract_current_sum,agg_all_contracts__g_contract__bit_da_guid__isMain__count__ALL_TIME,agg_all_contracts__g_contract__abs_change_price_last_ds__isMain__last__ALL_TIME,agg_all_contracts__g_contract__abs_change_price_last_ds__isMain__mean__ALL_TIME,agg_all_contracts__g_contract__rel_change_price_last_ds__isMain__last__ALL_TIME,agg_all_contracts__g_contract__rel_change_price_last_ds__isMain__mean__ALL_TIME,agg_cec_requests__g_contract__request_id__all__count__1W,agg_cec_requests__g_contract__request_id__all__count__2W,agg_cec_requests__g_contract__request_id__all__count__3W,...,agg_ArbitrationCases__g_contractor__PlaintiffSum__sum__12_48M,agg_ArbitrationCases__g_contractor__PlaintiffSum__sum__ALL_TIME,agg_tender_proposal__g_contractor__id__ALL__countDistinct__1W,agg_tender_proposal__g_contractor__id__ALL__countDistinct__2W,agg_tender_proposal__g_contractor__id__ALL__countDistinct__4W,agg_tender_proposal__g_contractor__id__ALL__countDistinct__8W,agg_tender_proposal__g_contractor__id__ALL__countDistinct__12W,agg_tender_proposal__g_contractor__id__ALL__countDistinct__26W,agg_tender_proposal__g_contractor__id__ALL__countDistinct__52W,agg_tender_proposal__g_contractor__id__ALL__countDistinct__ALL_TIME
0,1.115267,0.793952,-0.539030,0.00000,0.000000,0.0000,0.000000,-0.255766,0.341961,0.146159,...,-0.158471,-0.158473,-0.136894,-0.157822,-0.178045,-0.190966,-0.200348,-0.225724,-0.256704,-0.258454
1,1.608002,2.681675,0.743673,1.67389,1.731805,-0.0186,-0.021261,0.808046,0.662319,0.608890,...,-0.158471,-0.158473,-0.136894,-0.157822,-0.178045,-0.190966,-0.200348,-0.225724,-0.256704,-0.258454
2,-0.360764,-0.416432,-0.539030,0.00000,0.000000,0.0000,0.000000,-0.255766,-0.298754,-0.316572,...,-0.158471,-0.158473,-0.136894,-0.157822,-0.178045,-0.190966,-0.200348,-0.225724,-0.256704,-0.258454
3,-0.089303,-0.193827,-0.539030,0.00000,0.000000,0.0000,0.000000,-0.255766,0.021603,-0.085207,...,-0.158470,-0.158471,-0.136894,-0.157822,-0.178045,-0.190966,-0.200348,-0.225724,-0.256704,-0.258454
4,-0.515778,0.133446,0.743673,0.54505,0.550885,0.0000,0.000000,-0.255766,-0.298754,-0.316572,...,-0.158470,-0.158471,-0.136894,-0.157822,-0.178045,-0.190966,-0.200348,-0.225724,-0.256704,-0.258454


In [331]:
best_params_cat = np.load('models_parameters/best_params_cat.npy', allow_pickle=True).item()

In [332]:
X, Y = X_train.copy(), Y_train.copy()

In [333]:
kfold_roc_aucs = []
sk = StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)

for i, (train_idx, test_idx) in enumerate(sk.split(X, Y, groups)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    Y_train, Y_test = Y.iloc[train_idx], Y.iloc[test_idx]
    print('----------------------------------------------------------')
    
    ## CatBoost ##
    Cat_md = CatBoostClassifier(
        loss_function='Logloss',
        iterations=500,
        depth=best_params_cat['depth'],
        learning_rate=best_params_cat['learning_rate'],
        l2_leaf_reg=best_params_cat['l2_leaf_reg'],
        bagging_temperature=best_params_cat['bagging_temperature'],
        border_count=best_params_cat['border_count'],
        random_strength=best_params_cat['random_strength'],
        verbose=False,
        random_seed=42,
        task_type='CPU'
    ).fit(X_train, Y_train)

    cat_pred = Cat_md.predict_proba(X_test)[:, 1]
    cat_score = roc_auc_score(Y_test, cat_pred)
    kfold_roc_aucs.append(cat_score)
    print('Fold', i, '==> CatBoost oof ROC-AUC score is ==>', cat_score)
    # cat_pred_test = Cat_md.predict_proba(test_cv)[:, 1]    


----------------------------------------------------------
Fold 0 ==> CatBoost oof ROC-AUC score is ==> 0.7475511875511874
----------------------------------------------------------
Fold 1 ==> CatBoost oof ROC-AUC score is ==> 0.7961379838468193
----------------------------------------------------------
Fold 2 ==> CatBoost oof ROC-AUC score is ==> 0.74275766537812
----------------------------------------------------------
Fold 3 ==> CatBoost oof ROC-AUC score is ==> 0.7457873727310697
----------------------------------------------------------
Fold 4 ==> CatBoost oof ROC-AUC score is ==> 0.7265984422866669


In [334]:
np.array(kfold_roc_aucs).mean()

0.7517665303587726

In [42]:
X_train

,contract_current_sum,agg_spark_extended_report__g_contractor__CreditLimitSum__last__ALL_TIME
0,0.793952,-0.113779
1,2.681675,-0.113779
2,-0.416432,-0.113779
3,-0.193827,-0.149404
4,0.133446,-0.149404
...,...,...
28825,-0.511512,-0.151915
28826,-0.504509,-0.151915
28828,-0.515112,-0.151915
28829,-0.504737,-0.151915


In [71]:
X_train[['contract_init_sum', 'specialization_id', 'project_id', 'contractor_id']]

,contract_id,report_date,specialization_id,contract_init_sum,contract_date,project_id,building_id,contractor_id,contract_current_sum,agg_all_contracts__g_contract__bit_da_guid__isMain__count__ALL_TIME,...,agg_ArbitrationCases__g_contractor__PlaintiffSum__sum__12_48M,agg_ArbitrationCases__g_contractor__PlaintiffSum__sum__ALL_TIME,agg_tender_proposal__g_contractor__id__ALL__countDistinct__1W,agg_tender_proposal__g_contractor__id__ALL__countDistinct__2W,agg_tender_proposal__g_contractor__id__ALL__countDistinct__4W,agg_tender_proposal__g_contractor__id__ALL__countDistinct__8W,agg_tender_proposal__g_contractor__id__ALL__countDistinct__12W,agg_tender_proposal__g_contractor__id__ALL__countDistinct__26W,agg_tender_proposal__g_contractor__id__ALL__countDistinct__52W,agg_tender_proposal__g_contractor__id__ALL__countDistinct__ALL_TIME
0,5433,2023-01-01,18,1.115267,2022-05-10 21:00:00,43,701,438,0.793952,-0.539030,...,-0.158471,-0.158473,-0.136894,-0.157822,-0.178045,-0.190966,-0.200348,-0.225724,-0.256704,-0.258454
1,6875,2023-01-01,18,1.608002,2022-08-21 21:00:00,31,268,438,2.681675,0.743673,...,-0.158471,-0.158473,-0.136894,-0.157822,-0.178045,-0.190966,-0.200348,-0.225724,-0.256704,-0.258454
2,1476,2023-01-01,18,-0.360764,2022-10-17 21:00:00,31,268,438,-0.416432,-0.539030,...,-0.158471,-0.158473,-0.136894,-0.157822,-0.178045,-0.190966,-0.200348,-0.225724,-0.256704,-0.258454
3,4469,2023-01-01,12,-0.089303,2022-04-20 21:00:00,43,697,484,-0.193827,-0.539030,...,-0.158470,-0.158471,-0.136894,-0.157822,-0.178045,-0.190966,-0.200348,-0.225724,-0.256704,-0.258454
4,1330,2023-01-01,12,-0.515778,2022-03-02 21:00:00,49,224,484,0.133446,0.743673,...,-0.158470,-0.158471,-0.136894,-0.157822,-0.178045,-0.190966,-0.200348,-0.225724,-0.256704,-0.258454
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28826,5078,2023-07-23,21,-0.484227,2023-04-13 21:00:00,18,915,683,-0.504509,0.743673,...,0.000000,0.000000,-0.136894,-0.009243,-0.093062,-0.095692,-0.132309,0.179293,0.150820,0.148360
28827,3854,2023-07-23,21,-0.500003,2023-04-13 21:00:00,18,915,683,-0.524174,0.743673,...,0.000000,0.000000,-0.136894,-0.009243,-0.093062,-0.095692,-0.132309,0.179293,0.150820,0.148360
28828,5351,2023-07-23,21,-0.481102,2023-07-16 21:00:00,50,298,683,-0.515112,-0.539030,...,0.000000,0.000000,-0.136894,-0.009243,-0.093062,-0.095692,-0.132309,0.179293,0.150820,0.148360
28829,57,2023-07-23,21,-0.468450,2023-03-16 21:00:00,45,915,683,-0.504737,-0.539030,...,0.000000,0.000000,-0.136894,-0.009243,-0.093062,-0.095692,-0.132309,0.179293,0.150820,0.148360
